# AlexNet

In [2]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten

model = Sequential()

# Layer 1: Convolutional layer with 64 filters of size 11x11x3
model.add(Conv2D(filters=64, kernel_size=(11,11), strides=(4,4), padding='valid', activation='relu', input_shape=(224,224,3)))

# Layer 2: Max pooling layer with pool size of 3x3
model.add(MaxPooling2D(pool_size=(3,3), strides=(2,2)))

# Layer 3-5: 3 more convolutional layers with similar structure as Layer 1
model.add(Conv2D(filters=192, kernel_size=(5,5), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3), strides=(2,2)))
model.add(Conv2D(filters=384, kernel_size=(3,3), padding='same', activation='relu'))
model.add(Conv2D(filters=256, kernel_size=(3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3), strides=(2,2)))

# Layer 6: Fully connected layer with 4096 neurons
model.add(Flatten())
model.add(Dense(4096, activation='relu'))

# Layer 7: Fully connected layer with 4096 neurons
model.add(Dense(4096, activation='relu'))


# ResNet

In [4]:
from keras.models import Model
from keras.layers import Input, Conv2D, BatchNormalization, Activation, Add
from keras.layers import MaxPooling2D, GlobalAveragePooling2D, Dense

def residual_block(x, filters, kernel_size, strides=(1, 1)):
  shortcut = x

  # Main path
  x = Conv2D(filters, kernel_size, strides=strides, padding='same')(x)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  x = Conv2D(filters, kernel_size, padding='same')(x)
  x = BatchNormalization()(x)

  if strides != (1, 1) or shortcut.shape[-1] != filters:
    shortcut = Conv2D(filters, (1, 1), strides=strides, padding='same')(shortcut)
    shortcut = BatchNormalization()(shortcut)

  # Add main and shortcut paths
  x = Add()([x, shortcut])
  x = Activation('relu')(x)
  return x

# Input layer
inputs = Input(shape=(224, 224, 3))

# Initial convolutional layer
x = Conv2D(64, (7, 7), strides=(2, 2), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D((3, 3), strides=(2, 2))(x)

# Residual blocks
x = residual_block(x, 64, (3, 3))
x = residual_block(x, 64, (3, 3))
x = residual_block(x, 128, (3, 3), strides=(2, 2))
x = residual_block(x, 128, (3, 3))
x = residual_block(x, 256, (3, 3), strides=(2, 2))
x = residual_block(x, 256, (3, 3))
x = residual_block(x, 512, (3, 3), strides=(2, 2))
x = residual_block(x, 512, (3, 3))

# Global average pooling and classification
x = GlobalAveragePooling2D()(x)
outputs = Dense(1000, activation='softmax')(x)

# Create the model
model = Model(inputs=inputs, outputs=outputs)